In [1]:
import logging
# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

In [2]:
import HBPy
from HBPy.Molecule.Crystal import Crystal,Atom
from pathlib import Path
import os
from collections import defaultdict
import numpy
import matplotlib.pyplot as plt

/home/bulou/venv/ATOMOD/lib/python3.12/site-packages/e3nn/o3/_wigner.py:10: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  _Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))


cuequivariance or cuequivariance_torch is not available. Cuequivariance acceleration will be disabled.


In [3]:
status={
        'NP':True,
        'abtem':True,
        'feff':True,
        'atomic probability map':True,
        'optimization':False,
    }
config={
        'root_dir':'simul',
        'train':{
            'TEM_img_dir'      : "train/TEM",        # répertoire de stockage des images TEM
            'EXAFS_dir'        : "train/EXAFS",      # répertoire de stockage des spectres EXAFS
            'prob_maps_img_dir': "train/prob_maps",  # répertoire de stockage des images TEM
            'nfo_dir'          : "train/nfo",  # répertoire de stockage des images TEM
        },
        'abtem':{
            'status':status['abtem'],
            'dx':0.04,
            'dy':0.04,
            'dz':4.08/2,
            'energy':300e3,
            'focal spread':40,
            'semiangle cutoff':20,
            'defocus':200,
            'cell scale':1.1
            },
        'atomic presence probability map':{
            'ninter':{ # nombre d'intervalles entre deux positions atomiques
                'x':20,
                'y':20,
                'z':2
            },
            'sigma': .6  # en Å, largeur de la gaussienne ~ rayon atomique ou un peu moins
        },
        'atomic probability map':{
            'status':status['atomic probability map']
        },
        'NP':{
            'status':status['NP'],
            'seed':2,
            'structure':{
                'optimization':status['optimization'],
                #  'composition':['Pt','Co','Au'],
                # 'radius':3.5,
                # 'a':0.5*(3.55+3.92),
                'composition':['Pt','Co','Au','Pd','Rh'],
                'radius':5.0,
                'a':3.92,
            },
            'nvaccum':2.0,
        },
        'image':{
            'xmin':0.0,
            'xmax':0.0,
            'ymin':0.0,
            'ymax':0.0
            },
        'feff':{
            'status':status['feff'],
            'parameters':{
                'TITLE':'FEFF INPUT FILE',
                'DEBYE_TEMP': 190.0,
                'SCF_RADIUS': 5.0,
                'RPATH': 5.0,     # typique 2.2xdistance plus proches voisins. changer pour étudier la cvg des spectres
                'EXAFS' : 20.0,   # xkmax - default 20 ang.^-1
                'EDGE': {'Co':'K','Ni':'K','Ru':'K','Rh':'K','Pd':'K','Ir':'L3','Pt':'L3','Au':'L3'},
                'RMAX':8.0,
                'feff_dir':  '/home/bulou/ownCloud/Notebooks/M2P2_HEA/Home/Modelisation/ATOMOD/JFEFF/feff90/unix/',
                'input_save_dir':'./',
                'filename':'feff.inp',
                'list_pgm':['rdinp','atomic','dmdw','pot',
                            'opconsat', 
                            'screen',
                            'xsph',
                            'fms',
                            'mkgtr',
                            'path', 
                            'genfmt',
                            'ff2x',
                            'sfconv',
                            'compton',
                            'eels',
                            'ldos'
                            ]
            }
        }
    }


_______________________________________
# Etape 1 : construire la nanoparticules
 _______________________________________
#   Etape 1.1 : la structure

In [4]:
NP=Crystal()
NP.build(a=config['NP']['structure']['a'],
    radius=config['NP']['structure']['radius'],
    materials='NP')
NP.origin_at_mass_center()
logger.info(f"min={NP.qmin} max={NP.qmax}")
logger.info(f"Mass center={NP.MC}")
logger.info(f"Number of atoms={len(NP.atoms)}")

2026-06-17 15:57:15,799 [INFO] - <module>() - min=[-2.94 -2.94 -2.94] max=[2.94 2.94 2.94]
2026-06-17 15:57:15,801 [INFO] - <module>() - Mass center=[ 4.44089210e-16 -3.48927236e-16 -2.53765263e-16]
2026-06-17 15:57:15,802 [INFO] - <module>() - Number of atoms=28


 #   Etape 1.2 : la distribution chimique   

In [5]:
 NP.set_composition(config['NP']['structure']['composition'],seed=config['NP']['seed'])
directory=Path.cwd()/config['root_dir']/config['train']['nfo_dir']/"XYZ"
directory.mkdir(parents=True, exist_ok=True)

NP.save(prefix="NP",fmt='xyz',directory=directory)
    


#   Etape 1.3 : (optionnelle) l'optimisation structurale et/ou chimique

In [6]:
 if config['NP']['structure']['optimization']:
     NP.optimize_ase()
     logger.info(f"Optimization DONE!")

In [7]:
import py3Dmol
from pathlib import Path

In [8]:
chemin_fichier = Path(directory/"NP.xyz")
xyz_data = chemin_fichier.read_text(encoding="utf-8")
vue = py3Dmol.view(width=400, height=400)
vue.addModel(xyz_data, "xyz")
vue.setStyle({'sphere': {'colorscheme': 'Jmol', 'scale': 1.0}, 'spacefill': {}})
vue.zoomTo()
vue.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [9]:
if config['abtem']['status']:
        NP.abTEM(config)


2026-06-17 15:57:15,835 [INFO] - abTEM() - TEM images directory = simul/train/TEM
2026-06-17 15:57:15,836 [INFO] - abTEM() - Cell size: 0.0
2026-06-17 15:57:15,842 [INFO] - abTEM() - slice_thickness= 2.04  sampling=0.04
/home/bulou/venv/ATOMOD/lib/python3.12/site-packages/abtem/slicing.py:48: RuntimeWarning: invalid value encountered in scalar divide
  slice_thickness = (thickness / n,) * int(n)
2026-06-17 15:57:15,845 [INFO] - abTEM() - potential extansion: (0.0, 0.0)
2026-06-17 15:57:15,846 [INFO] - abTEM() - potential origin: (0.0, 0.0, 0.0)
2026-06-17 15:57:15,847 [INFO] - abTEM() - potential shape: (0, 0, 0)
/home/bulou/venv/ATOMOD/lib/python3.12/site-packages/abtem/waves.py:965: SyntaxWarning: invalid escape sequence '\e'
  tex_label="$Z, n, \ell$",


ZeroDivisionError: integer modulo by zero

In [ ]:
feff_dir=Path.cwd()/config['root_dir']/config['train']['nfo_dir']/"feff_input_files"
if config['feff']['status']:
    feff_dir.mkdir(parents=True, exist_ok=True)
    base_dir=os.getcwd()
    if config['feff']['status']:
        for atm in NP.atoms:
            config['feff']['parameters']['input_save_dir']=f"{feff_dir}/{atm.elt}_{atm.idx}"
            os.makedirs(config['feff']['parameters']['input_save_dir'], exist_ok=True)
            os.chdir(f"{config['feff']['parameters']['input_save_dir']}")
            NP.FEFF_create_input_file(config['feff']['parameters'],absorber_idx=atm.idx)
            NP.FEFF_run(config['feff']['parameters'])
            os.chdir(base_dir)
print("DONE!")

In [ ]:
# Initialise automatiquement avec une liste vide
series = defaultdict(list)
logger.info(NP.list_elt)
base_dir=os.getcwd()
logger.info(f"base directory: {base_dir}")
for atm in NP.atoms:
    xmu_dir = f"{feff_dir}/{atm.elt}_{atm.idx}"
    filepath = f"{xmu_dir}/xmu.dat"
    if not os.path.exists(filepath):
        logger.error(f"Le fichier n'existe pas du tout sur le disque : {filepath}")
        continue
    logger.info(f"xmu_dir={xmu_dir}")
    try:
        k, chi = numpy.loadtxt(filepath, comments='#', usecols=(2, 5), unpack=True)
        series[atm.elt].append((k, chi))
    except Exception as e:
        logger.error(f"Erreur de lecture dans {filepath}. Message d'erreur : {e}")
exafs_dir=Path.cwd()/config['root_dir']/config['train']['EXAFS_dir']
exafs_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
for elt in NP.list_elt:
    k,chi,dev=HBPy.Molecule.Tools.mk_mean(series[elt])
    series[elt].append((k,chi))

In [ ]:
# 1. On calcule la largeur idéale de l'image (par exemple 5 pouces par graphique)
largeur = 5 * len(NP.list_elt)
# 2. L'astuce : on donne [list_elt] à subplot_mosaic !
# Si list_elt = ['Fe', 'Ni'], cela crée la grille [['Fe', 'Ni']]
fig, ax = plt.subplot_mosaic([NP.list_elt], figsize=(largeur, 5))
color={}
color['Co']=(236/255,141/255,157/255)
color['Au']=(238/255,195/255,32/255)
color['Pt']=(178/255,178/255,193/255)
color['Rh']=(178/255,178/255,193/255)
color['Pd']=(178/255,178/255,193/255)

idx=3

if idx==0:
    by=0.15
    ylbl=r"$\chi(E)$"
    savename="chi(k)"
    header="# k (A^-1)   chi(k)"
elif idx==1:
    by=0.30
    ylbl=r"$k\cdot\chi(E)$"
    savename="kchi(k)"
    header="# k (A^-1)   k.chi(k)"
elif idx==2:
    by=1.0
    ylbl=r"$k^2\cdot\chi(E)$"
    savename="k2chi(k)"
    header="# k (A^-1)   k^2.chi(k)"
elif idx==3:
    by=5.0
    ylbl=r"$k^3\cdot\chi(E)$"
    savename="k3chi(k)"
    header="# k (A^-1)   k^3.chi(k)"

for elt in NP.list_elt:
    #for i, (k, chi) in enumerate(series[elt][:-1]):
        # L'utilisation de 'enumerate' donne un compteur 'i' (0, 1, 2...) très pratique pour la légende
    #    ax[elt].plot(k, chi*k**idx, label=f"Configuration {i+1}",color=color[elt])
    k,chiglob=series[elt][-1]
    ax[elt].plot(k,chiglob*k**idx, label=f"Configuration {i+1}",color='blue',linewidth=3)

    # Esthétique du graphique
    ax[elt].set_xlabel(r"$k \ (\AA^{-1})$")
    ax[elt].set_ylabel(ylbl) # Le 'r' permet de lire correctement les caractères spéciaux
    ax[elt].set_title(f"Spectre global pour l'élément {elt}")
    #plt.legend()
    ax[elt].grid(True, linestyle='--', alpha=0.6)
    ax[elt].set_xlim(2, 8)
    ax[elt].set_ylim(-by,by)

    numpy.savetxt(
        f"{savename}_{elt}.dat",          # Nom du fichier à créer
        numpy.column_stack((k, chiglob)),           # Les données en colonnes
        delimiter="   ",            # Séparateur (ici 3 espaces, ou '\t' pour une tabulation)
        header=header, # (Optionnel) Première ligne de texte
        comments="# ",              # (Optionnel) Symbole au début de l'entête (comme vos fichiers FEFF)
        fmt="%.6f"                  # (Optionnel) Format des nombres : ici des floats avec 6 décimales
    )
fig.subplots_adjust(wspace=0.4)
plt.savefig(
    f"{savename}.png",  # Nom du fichier (avec l'extension voulue)
    dpi=300,                 # Résolution (300 DPI = qualité impression/publication)
    bbox_inches='tight'      # Ajuste la boîte pour ne JAMAIS couper les labels ou les titres
)

plt.show()